In [34]:
import sys
import glob
import xarray as xr
import numpy as np
from matplotlib import pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature
import datetime
import geopandas as gpd
from shapely.geometry import mapping
from scipy.stats import spearmanr, pearsonr

import warnings
warnings.filterwarnings('ignore')

In [35]:
datap = "/Users/ellendyer/Documents/GitHub/Isotopes_F4R/plots/"
dataf = "/Users/ellendyer/Documents/GitHub/F4R_data/"

### Read in precip 
- do this for whole available ts and then sub-select years in next step for analysis

In [36]:
Y1=2018
Y2=2024

sif_all_list = []
for Y in range(Y1,Y2+1):
    pathlist = glob.glob(f'/Volumes/SIF/SIF_L3/S5P_PAL__L3__SIF____{Y}*')
    files_list = []
    for i in pathlist:
        print(i)
        sifin = xr.open_dataset(i,drop_variables=['datetime_stop'])
        #print(sifin)
        sifin = sifin.rename({'latitude':'lat','longitude':'lon','datetime_start':'time','solar_induced_fluorescence':'SIF'})
        sifin = sifin.assign_coords({"time": sifin.time})
        sifin = sifin['SIF']
        sifin = sifin.sel(lat=slice(-15,12),lon=slice(8,31),drop=True).load()
        files_list.append(sifin)
    sif = xr.concat(files_list,dim='time') 
    sif_year_list = []
    for m in range(1,13):
        try:
            print(m)
            mp = sif.sel(time=(sif.time.dt.month==m), drop=True)
            bins = [datetime.datetime(Y, m, 1),datetime.datetime(Y, m, 10),datetime.datetime(Y, m, 20),datetime.datetime(Y, m, mp.time.dt.days_in_month.values[0])]
            print(bins)
            mp_out = mp.groupby_bins('time', bins,labels=[datetime.datetime(Y, m, 10),datetime.datetime(Y, m, 20),datetime.datetime(Y, m, mp.time.dt.days_in_month.values[0])]).mean()
            mp_out = mp_out.rename({'time_bins':'time'})
            #print(mp_out)
            sif_year_list.append(mp_out)
        except:
            print('no month - ',m,' for year - ',Y)
    sif_year = xr.concat(sif_year_list,dim='time')
    sif_all_list.append(sif_year)
    sif.close()
    print('done - ',Y)
sif_all = xr.concat(sif_all_list,dim='time')
sif_all = sif_all.sel(time=slice('2018-07-01','2024-12-31'))
print(sif_all)
print(sif_all.time)
    
sif_all.to_netcdf(dataf+'sif_10day_reg_regrid.nc',engine='h5netcdf')
        

/Volumes/SIF/SIF_L3/S5P_PAL__L3__SIF____20180430T102851_20180430T121021_02824_03_010001_20230829T114646.nc
/Volumes/SIF/SIF_L3/S5P_PAL__L3__SIF____20180430T121021_20180430T135151_02825_03_010001_20230829T114603.nc
/Volumes/SIF/SIF_L3/S5P_PAL__L3__SIF____20180501T100953_20180501T115123_02838_03_010001_20230829T114550.nc
/Volumes/SIF/SIF_L3/S5P_PAL__L3__SIF____20180501T115123_20180501T133254_02839_03_010001_20230829T114529.nc
/Volumes/SIF/SIF_L3/S5P_PAL__L3__SIF____20180502T095056_20180502T113226_02852_03_010001_20230829T114432.nc
/Volumes/SIF/SIF_L3/S5P_PAL__L3__SIF____20180502T113226_20180502T131356_02853_03_010001_20230829T114501.nc
/Volumes/SIF/SIF_L3/S5P_PAL__L3__SIF____20180503T111328_20180503T125458_02867_03_010001_20230829T114410.nc
/Volumes/SIF/SIF_L3/S5P_PAL__L3__SIF____20180503T125458_20180503T143628_02868_03_010001_20230829T114231.nc
/Volumes/SIF/SIF_L3/S5P_PAL__L3__SIF____20180504T105430_20180504T123601_02881_03_010001_20230829T114253.nc
/Volumes/SIF/SIF_L3/S5P_PAL__L3__SIF_